In [ ]:
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install openpyxl
%pip install scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import openpyxl
import re
import warnings

warnings.filterwarnings('ignore')

# Estilo de los gráficos
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})
sns.set_palette('tab10')

# Paleta por centro de cultivo
CENTRO_COLORES = {
    'Cordova_2': '#2196F3',
    'Elena_Norte': '#4CAF50',
    'Concheo_1': '#FF9800',
}

In [ ]:
# Carpeta de salida
import os

CARPETA_FIGURAS = './resultados/graficos/exploracion_sensores'
os.makedirs(CARPETA_FIGURAS, exist_ok=True)

def guardar_figura(fig, nombre, dpi=300):
    ruta_png = os.path.join(CARPETA_FIGURAS, f'{nombre}.png')
    ruta_pdf = os.path.join(CARPETA_FIGURAS, f'{nombre}.pdf')
    fig.savefig(ruta_png, dpi=dpi, bbox_inches='tight')
    fig.savefig(ruta_pdf, bbox_inches='tight')
    print(f'Guardado: {ruta_png}  |  {ruta_pdf}')


In [ ]:
RUTA_EXCEL = os.environ.get('SENSOR_DATASET_PATH', './dataset_sensores/Stats_Extraidos.xlsx')

df = pd.read_excel(RUTA_EXCEL, sheet_name='Todos')
df.head(3)

In [ ]:
# Análisis de datos faltantes
resumen = pd.DataFrame({
    'dtype':  df.dtypes,
    'nulos':  df.isnull().sum(),
    '% nulo': (df.isnull().mean() * 100).round(2)
})
resumen

In [ ]:
PATRON_FECHA = r'(\d{2}/\d{2}/\d{4} \d{2}:\d{2}:\d{2})'

fechas = df['Periodo'].str.extractall(PATRON_FECHA)[0].unstack('match')
fechas.columns = ['Fecha_inicio_str', 'Fecha_fin_str']

df['Fecha_inicio'] = pd.to_datetime(fechas['Fecha_inicio_str'], format='%d/%m/%Y %H:%M:%S')
df['Fecha_fin']    = pd.to_datetime(fechas['Fecha_fin_str'],    format='%d/%m/%Y %H:%M:%S')
df['Duracion_h']   = (df['Fecha_fin'] - df['Fecha_inicio']).dt.total_seconds() / 3600

print(f"- Inicio: {df['Fecha_inicio'].min()}")
print(f"- Fin:    {df['Fecha_fin'].max()}")

df[['Archivo','Equipo','Fecha_inicio','Fecha_fin','Duracion_h']].head(4)

In [ ]:
def extraer_profundidad(equipo_str):
    m = re.search(r'Sensor ([\d,\.]+)\s*mts', equipo_str, re.IGNORECASE)
    if m:
        return float(m.group(1).replace(',', '.'))
    return np.nan

def extraer_ubicacion(equipo_str):
    m = re.search(r'(Jaula\s*\d+|Pont[oó]n)', equipo_str, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return 'Desconocido'

df['Profundidad_m'] = df['Equipo'].apply(extraer_profundidad)
df['Ubicacion']     = df['Equipo'].apply(extraer_ubicacion)
df = df[df['Profundidad_m'].isin([5, 10])].copy() #Filtramos solo profundidades de 5 y 10 metros ya que se comparten entre los 3 centros

print('Distribución de profundidades:')
print(df['Profundidad_m'].value_counts().sort_index())
print(f"\nRegistros sin profundidad: {df['Profundidad_m'].isna().sum()}")

In [ ]:
df['Equipo_norm'] = (
    df['Equipo']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.replace('Ponton', 'Pontón', regex=False)
)

In [ ]:
# Fecha sintética (mes-día) para comparar estacionalidad
# independientemente del año de registro de cada centro.

AÑO_REF = 2021  # año bisiesto arbitrario de referencia (permite 29 feb)

if 'Fecha_inicio' not in df.columns or 'Fecha_fin' not in df.columns:
    PATRON_FECHA = r'(\d{2}/\d{2}/\d{4} \d{2}:\d{2}:\d{2})'
    fechas = df['Periodo'].str.extractall(PATRON_FECHA)[0].unstack('match')
    fechas.columns = ['Fecha_inicio_str', 'Fecha_fin_str']

    df['Fecha_inicio'] = pd.to_datetime(
        fechas['Fecha_inicio_str'], format='%d/%m/%Y %H:%M:%S'
    )
    df['Fecha_fin'] = pd.to_datetime(
        fechas['Fecha_fin_str'], format='%d/%m/%Y %H:%M:%S'
    )
    df['Duracion_h'] = (df['Fecha_fin'] - df['Fecha_inicio']).dt.total_seconds() / 3600

# Reemplazamos el año original por el de referencia
def a_fecha_sintetica(fecha):
    try:
        return fecha.replace(year=AÑO_REF)
    except ValueError:
        # Maneja 29 de febrero en años no bisiestos -> usa 28 feb
        return fecha.replace(year=AÑO_REF, day=28)

df['Fecha_sintetica'] = df['Fecha_inicio'].apply(a_fecha_sintetica)

# Columna auxiliar de mes (1-12) y nombre de mes, útil para
# agrupar/filtrar independientemente del día exacto
df['Mes'] = df['Fecha_inicio'].dt.month
df['Mes_nombre'] = df['Fecha_inicio'].dt.strftime('%b')

print("Rango de fechas sintéticas (año de referencia ficticio):")
print(f"- Mínimo: {df['Fecha_sintetica'].min()}")
print(f"- Máximo: {df['Fecha_sintetica'].max()}")
print()
print("Meses disponibles por centro de cultivo:")
print(df.groupby('Archivo')['Mes'].apply(lambda x: sorted(x.unique())))

In [ ]:
VARS_PROM = ['O2_Prom', 'Temp_Prom', 'Sat_Prom', 'Sal_Prom']
VARS_DE   = ['O2_DE', 'Temp_DE', 'Sat_DE', 'Sal_DE']

df_raw = df.copy()

# df pasa a ser el PROMEDIO DIARIO por centro de cultivo: se
# promedian todos los sensores, equipos y profundidades (5 y 10 m)
# disponibles ese día. Esto para hacerlos comparables entre sí.
df = (
    df_raw
    .groupby(['Archivo', 'Fecha_sintetica'], dropna=False)[VARS_PROM + VARS_DE]
    .mean()
    .reset_index()
)
df['Mes'] = df['Fecha_sintetica'].dt.month
df['Mes_nombre'] = df['Fecha_sintetica'].dt.strftime('%b')

print(f"df_raw: {len(df_raw)} filas (una por sensor/periodo (profundidades 5 y 10 m))")
print(f"df:     {len(df)} filas (una por centro y día sintético"
      f"(promedio de sensores y profundidades de ese día))")
df.head()


In [ ]:
df.groupby('Archivo')[VARS_PROM].mean().round(2)

In [ ]:
# Histogramas: una figura individual por variable
etiquetas = {
    'O2_Prom':   'O2 promedio (mg/L)',
    'Temp_Prom': 'Temperatura promedio (°C)',
    'Sat_Prom':  'Saturación promedio (%)',
    'Sal_Prom':  'Salinidad promedio (PSU)',
}

N_BINS = 40

for var, etiqueta in etiquetas.items():
    datos_global = df[var].dropna()
    bin_edges = np.linspace(datos_global.min(), datos_global.max(), N_BINS + 1)

    fig, ax = plt.subplots(figsize=(6, 4.5))
    for camp, color in CENTRO_COLORES.items():
        subset = df[df['Archivo'] == camp][var].dropna()
        ax.hist(subset, bins=bin_edges, alpha=0.6, color=color,
                label=camp.replace('_', ' '))
    ax.set_xlabel(etiqueta)
    ax.set_ylabel('Frecuencia')
    ax.set_title(f'Distribución de {etiqueta}', fontsize=11)
    ax.legend(fontsize=8)
    plt.tight_layout()
    guardar_figura(fig, f'histograma_{var}')
    plt.show()


In [ ]:
for var, etiqueta in etiquetas.items():
    datos_global = df[var].dropna()
    bin_edges = np.linspace(datos_global.min(), datos_global.max(), N_BINS + 1)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=False)
    for ax, (camp, color) in zip(axes, CENTRO_COLORES.items()):
        subset = df[df['Archivo'] == camp][var].dropna()
        ax.hist(subset, bins=bin_edges, color=color, alpha=0.8, density=True)
        ax.set_title(camp.replace('_', ' '), fontsize=10)
        ax.set_xlabel(etiqueta)
    axes[0].set_ylabel('Densidad')
    fig.suptitle(f'Distribución de {etiqueta} por centro',
                 fontsize=11)
    plt.tight_layout()
    guardar_figura(fig, f'histograma_panel_{var}')
    plt.show()


In [ ]:
# Detección de outliers utilizando IQR
def flag_iqr(series, factor=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return (series < q1 - factor * iqr) | (series > q3 + factor * iqr)

for var in VARS_PROM:
    df[f'{var}_outlier'] = flag_iqr(df[var])

# Resumen de flags
flags = [f'{v}_outlier' for v in VARS_PROM]
resumen_out = df[flags].sum().rename(lambda x: x.replace('_outlier',''))
print('Outliers detectados por variable:')
print(resumen_out.to_string())

df['Outlier_alguno'] = df[flags].any(axis=1)
print(f"\nRegistros con al menos un outlier: {df['Outlier_alguno'].sum()} "
      f"({df['Outlier_alguno'].mean()*100:.1f}%)")

In [ ]:
# Boxplots: una figura individual por variable.
for var, etiqueta in etiquetas.items():
    fig, ax = plt.subplots(figsize=(6, 4.5))
    data = [df[df['Archivo'] == c][var].dropna().values
            for c in CENTRO_COLORES]
    bp = ax.boxplot(data, patch_artist=True, notch=False,
                    flierprops=dict(marker='o', markersize=3, alpha=0.4))
    for patch, color in zip(bp['boxes'], CENTRO_COLORES.values()):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_xticklabels(
        [c.replace('_', ' ') for c in CENTRO_COLORES],
        rotation=15
    )
    ax.set_ylabel(etiqueta)
    ax.set_title(f'Distribución de {etiqueta} por centro de cultivo', fontsize=11)
    plt.tight_layout()
    guardar_figura(fig, f'boxplot_{var}')
    plt.show()


In [ ]:
# Correlación entre variables ambientales (a nivel de promedio
# diario por centro).
vars_corr = VARS_PROM + VARS_DE
corr = df[vars_corr].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, linewidths=0.5, ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Correlaciones entre variables (promedio diario)', fontsize=12)
plt.tight_layout()
guardar_figura(fig, 'heatmap_correlacion_variables')
plt.show()


In [ ]:
# Series de tiempo por centro de cultivo, alineadas por mes-día
# (independiente del año de registro real de cada centro).
for var, etiqueta in etiquetas.items():
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for camp, color in CENTRO_COLORES.items():
        sub = df[df['Archivo'] == camp].sort_values('Fecha_sintetica')
        serie = sub.set_index('Fecha_sintetica')[var].resample('D').mean()
        serie = serie.interpolate(limit=3, limit_direction='both') # Si hay algún Nan se interpola entre los valores cercanos.

        if serie.dropna().empty:
            continue
        tramo = []
        for idx, valor in serie.items():
            if pd.notna(valor):
                tramo.append((idx, valor))
            elif tramo:
                x_tramo = [t[0] for t in tramo]
                y_tramo = [t[1] for t in tramo]
                ax.plot(x_tramo, y_tramo, alpha=0.35, color=color, linewidth=0.8)
                roll = pd.Series(y_tramo, index=x_tramo).rolling(7, min_periods=3).mean()
                ax.plot(roll.index, roll.values, color=color, linewidth=1.8)
                tramo = []

        if tramo:
            x_tramo = [t[0] for t in tramo]
            y_tramo = [t[1] for t in tramo]
            ax.plot(x_tramo, y_tramo, alpha=0.35, color=color, linewidth=0.8)
            roll = pd.Series(y_tramo, index=x_tramo).rolling(7, min_periods=3).mean()
            ax.plot(roll.index, roll.values, color=color, linewidth=1.8, label=camp.replace('_', ' '))

    ax.set_ylabel(etiqueta)
    ax.set_xlabel('Mes del año')
    ax.set_title(
        f'Serie de tiempo de {etiqueta} por centro de cultivo\n',
        fontsize=11
    )
    ax.legend(fontsize=8, loc='upper right')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    plt.tight_layout()
    guardar_figura(fig, f'serie_tiempo_{var}')
    plt.show()


In [ ]:
# Mapeo de centro de cultivo a región
# Disponible por si se desea retomar la comparación agregada por
# región en el futuro. Los tests de significancia de las celdas
# siguientes están enfocados únicamente en los 3 centros de cultivo
# y no requieren esta columna.

REGION_POR_CENTRO = {
    'Cordova_2':   'Magallanes',
    'Elena_Norte': 'Aysen',
    'Concheo_1':   'Aysen',
}

df['Region'] = df['Archivo'].map(REGION_POR_CENTRO)
df['Region'].value_counts()


## Tests de significancia estadística

In [ ]:
# Funciones auxiliares para tests de significancia
# (comparación entre centros de cultivo)
from scipy import stats
from itertools import combinations

# Se aplica Shapiro-Wilk a cada grupo.
# Devuelve True si todos los grupos pueden considerarse normales (p > alpha).
def verificar_normalidad(grupos_dict, alpha=0.05):
    print(f"{'Grupo':<20} {'N':>6} {'Shapiro W':>10} {'p-value':>10} {'¿Normal?':>10}")
    todos_normales = True
    for nombre, datos in grupos_dict.items():
        datos = pd.Series(datos).dropna()
        stat, p = stats.shapiro(datos)
        es_normal = p > alpha
        todos_normales &= es_normal
        print(f"{nombre:<20} {len(datos):>6} {stat:>10.4f} {p:>10.4g} {str(es_normal):>10}")
    return todos_normales

# Test de levene para comprobar homogeneidad de varianzas entre grupos.
def verificar_homogeneidad_varianzas(grupos_dict, alpha=0.05):
    datos = [pd.Series(v).dropna() for v in grupos_dict.values()]
    stat, p = stats.levene(*datos)
    print(f"Levene: estadístico = {stat:.4f}, p-value = {p:.4g} "
          f"-> {'Varianzas homogéneas' if p > alpha else 'Varianzas NO homogéneas'}")
    return p > alpha

# Aplicar ANOVA de una vía si se cumplen los supuestos, o Kruskal Wallis en caso contrario.
def test_omnibus_centros(grupos_dict, es_normal, var_homogenea, alpha=0.05):
    datos = [pd.Series(v).dropna() for v in grupos_dict.values()]

    if es_normal and var_homogenea:
        stat, p = stats.f_oneway(*datos)
        nombre_test = "ANOVA de una vía"
    else:
        stat, p = stats.kruskal(*datos)
        nombre_test = "Kruskal-Wallis (no paramétrico)"

    print(f"\n>>> Test omnibus aplicado: {nombre_test}")
    print(f"    Estadístico = {stat:.4f}, p-value = {p:.4g}")
    print(f"    {'Diferencia SIGNIFICATIVA' if p < alpha else 'Sin diferencia significativa'} (alpha={alpha})")
    return nombre_test, stat, p

# Comparación por pares con correción de Bonferroni (t-test (Welch) si se cumple la normalidad, o Mann-Whitney U si no).
def post_hoc_pares(grupos_dict, es_normal, alpha=0.05):
    nombres = list(grupos_dict.keys())
    pares = list(combinations(nombres, 2))
    n_comparaciones = len(pares)
    alpha_corregido = alpha / n_comparaciones

    print(f"\n>>> Post-hoc por pares (corrección de Bonferroni, "
          f"alpha ajustado = {alpha_corregido:.4g})")
    resultados = []
    for g1, g2 in pares:
        d1 = pd.Series(grupos_dict[g1]).dropna()
        d2 = pd.Series(grupos_dict[g2]).dropna()
        if es_normal:
            stat, p = stats.ttest_ind(d1, d2, equal_var=False)
            test_usado = "Welch t-test"
        else:
            stat, p = stats.mannwhitneyu(d1, d2, alternative='two-sided')
            test_usado = "Mann-Whitney U"
        sig = "***" if p < alpha_corregido else "ns"
        resultados.append({
            'Comparación': f'{g1} vs {g2}',
            'Test': test_usado,
            'Estadístico': round(stat, 4),
            'p-value': p,
            'Significativo (Bonferroni)': sig
        })
        print(f"  {g1:<22} vs {g2:<22} -> p = {p:.4g}  [{sig}]")

    return pd.DataFrame(resultados)

# Se aplica el flujo completo de pruebas:
# normalidad -> homogeneidad de varianzas -> test omnibus -> post-hoc
def aplicar_test_completo(df, var, centros=None):
    if centros is None:
        centros = df['Archivo'].unique()

    grupos = {c: df[df['Archivo'] == c][var].dropna() for c in centros}

    print("=" * 70)
    print(f"COMPARACIÓN POR CENTRO DE CULTIVO — {var}")
    print("=" * 70)
    normal = verificar_normalidad(grupos)
    print()
    var_homog = verificar_homogeneidad_varianzas(grupos)
    nombre_test, stat, p = test_omnibus_centros(grupos, normal, var_homog)

    tabla_posthoc = None
    if p < 0.05:
        tabla_posthoc = post_hoc_pares(grupos, normal)
    else:
        print("\nNo se realiza post-hoc: el test omnibus no fue significativo.")

    return {
        'variable': var,
        'test_usado': nombre_test,
        'estadistico': stat,
        'p_value': p,
        'posthoc': tabla_posthoc,
    }


### Aplicación a las cuatro variables ambientales

In [ ]:
VARIABLES = ['Temp_Prom', 'Sal_Prom', 'O2_Prom', 'Sat_Prom']

resultados_centros = {}
for var in VARIABLES:
    resultados_centros[var] = aplicar_test_completo(df, var)
    print("\n")


### Tabla resumen de resultados

In [ ]:
filas_resumen = []
for var, res in resultados_centros.items():
    if res['posthoc'] is not None:
        no_sig = res['posthoc'][res['posthoc']['Significativo (Bonferroni)'] == 'ns']['Comparación'].tolist()
        pares_no_sig = ', '.join(no_sig) if no_sig else 'Ninguno'
    else:
        pares_no_sig = 'N/A (test omnibus no significativo)'

    filas_resumen.append({
        'Variable': var,
        'Test omnibus': res['test_usado'],
        'p-value': f"{res['p_value']:.4g}" if res['p_value'] > 0 else '< 1e-300',
        'Pares no significativos': pares_no_sig,
    })

tabla_resumen = pd.DataFrame(filas_resumen)
tabla_resumen
